<a href="https://colab.research.google.com/github/LucasCAraujo21/Analisador-L-xico/blob/main/Analisador_lexico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [45]:
%pip install -q lark ipywidgets
import html
import lark
from lark import Lark
from lark.exceptions import UnexpectedCharacters, UnexpectedInput
import ipywidgets as widgets
from IPython.display import display, HTML
from collections import Counter

print(f" Lark versão {lark.__version__} pronto!")
print(f" ipywidgets versão {widgets.__version__} pronto!")


 Lark versão 1.3.1 pronto!
 ipywidgets versão 7.7.1 pronto!


In [50]:
# Paleta de cores: cada categoria de token ganha uma cor.
CORES = {

    # --- Palavras reservadas / palavras-chave (azul) ---
    "COMPRAR": "#1565c0",
    "INGRESSO": "#1565c0",
    "EVENTO": "#1565c0",
    "TIPO": "#1565c0",
    "TIPO_ESCOLHIDO": "#1565c0",
    "CONFIRMAR": "#1565c0",
    "CANCELAR": "#1565c0",
    "SETOR": "#1565c0",
    "SETOR_ESCOLHIDO": "#1565c0",
    "LOTE": "#1565c0",
    "PAGAMENTO": "#1565c0",
    "FORMA_PGTO": "#1565c0",
    "EM": "#1565c0",
    "AS": "#1565c0",
    "NOME": "#1565c0",

    # --- Literais / valores numéricos (verde / laranja) ---
    "DATA": "#ef6c00",
    "HORA": "#ef6c00",
    "PRECO": "#2e7d32",
    "QTD": "#2e7d32",
    "NUMERO": "#2e7d32",

    # --- Textos e identificadores (ciano / vermelho) ---
    "ITEM": "#00838f",
    "STRING": "#00838f",
    "PALAVRA": "#00838f",

    # --- Outros tokens ---
    "ID": "#00838f",
    "DOIS_PONTOS": "#c62828",
    "VIRGULA": "#c62828",

    # --- Chaves PIX ---
    "CHAVE_EMAIL": "#ad1457",
    "CHAVE_CPF": "#ad1457",
    "CHAVE_CNPJ": "#ad1457",
    "CHAVE_TELEFONE": "#ad1457",
    "CHAVE_ALEATORIA": "#ad1457",
}

def cor_do_token(tipo):
    """Devolve a cor da categoria (verde se não estiver na paleta)."""
    return CORES.get(tipo, "#00FF00")

def tabela_tokens_html(tokens, titulo="Tabela de Tokens"):
    """Monta uma tabela HTML com: nº, TOKEN (categoria), LEXEMA, linha e coluna."""
    linhas = ""
    for i, t in enumerate(tokens, start=1):
        cor = cor_do_token(t.type)
        linhas += (
            f"<tr><td>{i}</td>"
            f"<td><b style='color:{cor}'>{t.type}</b></td>"
            f"<td><code>{html.escape(str(t.value))}</code></td>"
            f"<td>{t.line}</td><td>{t.column}</td></tr>"
        )
    return f"""
    <h4 style='margin:4px 0'>{titulo} ({len(tokens)} tokens)</h4>
    <table style='border-collapse:collapse;font-family:monospace;font-size:13px'>
      <tr style='background:#263238;color:white'>
        <th style='padding:4px 10px'>#</th><th style='padding:4px 10px'>TOKEN</th>
        <th style='padding:4px 10px'>LEXEMA</th><th style='padding:4px 10px'>LINHA</th>
        <th style='padding:4px 10px'>COLUNA</th>
      </tr>
      {linhas}
    </table>"""

def texto_colorido_html(texto, tokens, transformar=None):
    """Pinta cada lexema no texto original, como faz o 'syntax highlight' do VS Code.
    Usa t.start_pos e t.end_pos: a posição exata de cada token no texto.
    'transformar' (opcional) troca o lexema exibido — usado para mascarar dados (LGPD)."""
    saida, cursor = "", 0
    for t in tokens:
        saida += html.escape(texto[cursor:t.start_pos])        # espaços/comentários
        cor = cor_do_token(t.type)
        saida += (f"<span title='{t.type}' style='color:{cor};font-weight:bold;"
                  f"border-bottom:2px solid {cor}'>"
                  f"{html.escape(transformar(t) if transformar else texto[t.start_pos:t.end_pos])}</span>")
        cursor = t.end_pos
    saida += html.escape(texto[cursor:])
    return ("<pre style='background:#fafafa;border:1px solid #ddd;padding:10px;"
            f"font-size:14px;line-height:1.7'>{saida}</pre>"
            "<small>💡 Passe o mouse sobre um lexema para ver o nome do token.</small>")

def erro_lexico_html(texto, erro):
    """Mostra um erro léxico com linha, coluna e uma 'setinha' apontando o problema."""
    linha_txt = texto.splitlines()[erro.line - 1] if texto.splitlines() else ""
    seta = " " * (erro.column - 1) + "^"
    return f"""
    <div style='background:#ffebee;border-left:5px solid #c62828;padding:10px'>
      <b>❌ ERRO LÉXICO</b> na linha <b>{erro.line}</b>, coluna <b>{erro.column}</b>:
      caractere inesperado <code>{html.escape(repr(erro.char))}</code>
      <pre style='margin:6px 0'>{html.escape(linha_txt)}\n{seta}</pre>
      <small>{html.escape(getattr(erro, 'dica', ''))}</small>
    </div>"""

print("✅ Ferramentas de visualização carregadas!")

✅ Ferramentas de visualização carregadas!


SOLUÇÃO

In [42]:
gramatica_a1 = r"""
// ---------- Regra inicial ----------
start: _token*
_token: COMPRAR | INGRESSO | EVENTO | TIPO | TIPO_ESCOLHIDO | FORMA_PGTO | PAGAMENTO | CONFIRMAR | CANCELAR
      | AS | EM | DATA | HORA | PRECO | QTD | ITEM | NOME | COMPRADOR | SETOR_ESCOLHIDO | SETOR | LOTE | NUMERO

// ---------- 1) Palavras reservadas ----------
FORMA_PGTO.3: /(pix|cart[aã]o)\b/i
PAGAMENTO.3: /pagamento\b/i
TIPO.3:      /tipo\b/i
TIPO_ESCOLHIDO.3: /(meia|inteira)\b/i
COMPRAR.3:  /comprar\b/i
INGRESSO.3: /ingresso\b/i
EVENTO.3:   /evento\b/i
CONFIRMAR.3: /confirmar\b/i
CANCELAR.3: /cancelar\b/i
SETOR.3:    /setor\b/i
SETOR_ESCOLHIDO.3: /(pista|banco)\b/i
LOTE.3:     /lote\b/i
EM.3:      /em\b/i
AS.3:      /as\b/i
NOME.3:     /nome\b/i
COMPRADOR: /[A-Za-zÀ-ÿ]+/


// ---------- 2) Literais ----------
QTD.2:  /\d+x/              // 2x, 10x  (prioridade 2: vence NUMERO se existir e evita ambiguidade)
DATA.2:  /\d{2}\/\d{2}\/\d{4}/ // 10/09/2026  (prioridade 2: vence NUMERO se existir)
HORA.2: /\d{2}:\d{2}/ // 14:32
PRECO.2: /R\$ ?\d{1,3}(\.\d{3})*,\d{2}/   // R$ 25,90
ITEM: /"[^"]+"/
NUMERO: /\d+/ // específico para número do lote


// ---------- 3) Ruído (descartado) ----------
COMENTARIO: /#[^\n]*/
%ignore COMENTARIO
%ignore /[ \t\r\n]+/
"""

# lexer="basic" -> o Lark usa um scanner clássico (igual Lex/Flex):
# lê o texto da esquerda para a direita e devolve tokens, um por vez.
lexer_a1 = Lark(gramatica_a1, parser="lalr", lexer="basic")

# Dicas amigáveis para os erros mais comuns (o "Stack Overflow" do nosso lexer)
DICAS_A1 = {
    #textos
    '"': 'Parece que você abriu aspas e não fechou. Todo texto precisa de "abre" e "fecha".',
    "'": 'Use aspas duplas em vez de aspas simples, por exemplo: "Show Slayer".',

    #preço
    "$": 'Preço inválido. Informe o preço com "R$", por exemplo: R$ 25,90.',
    ".": 'Verifique o preço. Use vírgula nos centavos: R$ 25,90.',
    ",": 'Verifique o formato do preço. Exemplo válido: R$ 25,90.',

    # quantidade
    "x": 'A quantidade deve ser informada como número seguido de "x", por exemplo: 2x ou 10x.',

}

def tokenizar_a1(texto):
    """Tokeniza o pedido. Se houver erro léxico, anexa uma dica ao erro."""
    try:
        return list(lexer_a1.lex(texto))
    except UnexpectedCharacters as erro:
        erro.dica = DICAS_A1.get(erro.char, f"Caractere {erro.char!r} não pertence ao alfabeto da linguagem de pedidos.")
        raise

def valor_do_preco(lexema):
    """Converte o LEXEMA 'R$ 1.250,90' no VALOR 1250.90 (atributo do token)."""
    return float(lexema.replace("R$", "").strip().replace(".", "").replace(",", "."))

# ---- Teste rápido no console ----
pedido = '''#compra de ingressos - sympla
INGRESSO 2x "Show Slayer" SETOR pista LOTE 2 tipo meia nome Lucas Carvalho de Araújo R$ 200,00 EM 10/12/2026
PAGAMENTO pix'''
print(f"Entrada: {pedido}\n")
for t in tokenizar_a1(pedido):
  print(f"L{t.line:<2} C{t.column:<3} {t.type:<10} {t.value!r}")

Entrada: #compra de ingressos - sympla
INGRESSO 2x "Show Slayer" SETOR pista LOTE 2 tipo meia nome Lucas Carvalho de Araújo R$ 200,00 EM 10/12/2026
PAGAMENTO pix

L2  C1   INGRESSO   'INGRESSO'
L2  C10  QTD        '2x'
L2  C13  ITEM       '"Show Slayer"'
L2  C27  SETOR      'SETOR'
L2  C33  SETOR_ESCOLHIDO 'pista'
L2  C39  LOTE       'LOTE'
L2  C44  NUMERO     '2'
L2  C46  TIPO       'tipo'
L2  C51  TIPO_ESCOLHIDO 'meia'
L2  C56  NOME       'nome'
L2  C61  COMPRADOR  'Lucas'
L2  C67  COMPRADOR  'Carvalho'
L2  C76  COMPRADOR  'de'
L2  C79  COMPRADOR  'Araújo'
L2  C86  PRECO      'R$ 200,00'
L2  C96  EM         'EM'
L2  C99  DATA       '10/12/2026'
L3  C1   PAGAMENTO  'PAGAMENTO'
L3  C11  FORMA_PGTO 'pix'


Interface com ipywidgets

In [ ]:
exemplos_a1 = {
    "Pedido simples": 'COMPRAR 2x "Show Slayer" R$ 900,90 SETOR pista',
    "Pedido meia": 'COMPRAR 1x "Show Korn" R$ 150,90 SETOR banco',
    "Pedido com data": 'COMPRAR 2x "Show Slayer" R$ 900,90 SETOR pista EM 10/12/2026',
    "Pedido com hora": 'COMPRAR 2x "Show Slayer" R$ 900,90 SETOR pista AS 14:32',
    "❌ Com erro (@)": 'COMPRAR 2x "Show Slayer" R$ 900,90 SETOR @pista',
}

entrada_a1 = widgets.Text(value=exemplos_a1["Pedido simples"],
                          description="Pedido:", layout=widgets.Layout(width="95%"))
seletor_a1 = widgets.Dropdown(options=list(exemplos_a1), description="Exemplos:")
botao_a1 = widgets.Button(description="🔍 Tokenizar", button_style="primary")
saida_a1 = widgets.Output()

def ao_escolher_exemplo_a1(mudanca):
    entrada_a1.value = exemplos_a1[mudanca["new"]]

def ao_clicar_a1(_):
    saida_a1.clear_output()
    with saida_a1:
        texto = entrada_a1.value
        try:
            tokens = tokenizar_a1(texto)
            display(HTML(texto_colorido_html(texto, tokens)))
            display(HTML(tabela_tokens_html(tokens)))
        except UnexpectedCharacters as erro:
            display(HTML(erro_lexico_html(texto, erro)))

seletor_a1.observe(ao_escolher_exemplo_a1, names="value")
botao_a1.on_click(ao_clicar_a1)

display(widgets.HTML("<h3>🎶 Comanda Digital — Analisador Léxico</h3>"),
        seletor_a1, entrada_a1, botao_a1, saida_a1)
ao_clicar_a1(None)   # já mostra o primeiro resultado

HTML(value='<h3>🎶 Comanda Digital — Analisador Léxico</h3>')

Dropdown(description='Exemplos:', options=('Pedido simples', 'Pedido meia', 'Pedido com data', 'Pedido com hor…

Text(value='COMPRAR 2x "Show Slayer" R$ 900,90 SETOR pista', description='Pedido:', layout=Layout(width='95%')…

Button(button_style='primary', description='🔍 Tokenizar', style=ButtonStyle())

Output()

Interface completa

In [51]:
def montar_recibo(tokens):
    itens = []
    pagamento = "não informado"
    obs = []

    i = 0

    while i < len(tokens):
        t = tokens[i]

        # Encontrou a quantidade de ingressos
        if t.type == "QTD":

            qtd = int(t.value[:-1])

            # Procura o ITEM logo depois da quantidade
            nome = None
            preco = None
            data = None

            j = i + 1

            while j < len(tokens):

                # Se chegou em outro ingresso, para
                if tokens[j].type == "INGRESSO":
                    break

                # Nome do evento
                if tokens[j].type == "ITEM" and nome is None:
                    nome = tokens[j].value.strip('"')

                # Preço
                elif tokens[j].type == "PRECO" and preco is None:
                    preco = valor_do_preco(tokens[j].value)

                # Data
                elif tokens[j].type == "DATA":
                    data = tokens[j].value

                # Pagamento
                elif tokens[j].type == "PAGAMENTO":
                    break

                j += 1

            # Só adiciona se encontrou nome e preço
            if nome is not None and preco is not None:
                item_data = {
                    "qtd": qtd,
                    "nome": nome,
                    "preco": preco
                }

                if data:
                    item_data["data"] = data

                itens.append(item_data)

            i = j
            continue

        # Forma de pagamento
        elif t.type == "PAGAMENTO":
            if i + 1 < len(tokens) and tokens[i + 1].type == "FORMA_PGTO":
                pagamento = tokens[i + 1].value.upper()
                i += 2
                continue

        i += 1

    # Soma o valor de todos os ingressos
    total = sum(
        item["qtd"] * item["preco"]
        for item in itens
    )

    brl = lambda v: (
        f"R$ {v:,.2f}"
        .replace(",", "X")
        .replace(".", ",")
        .replace("X", ".")
    )

    linhas = "".join(
        f"""
        <tr>
            <td>{item['qtd']}x</td>
            <td>{html.escape(item['nome'])}</td>
            <td align='right'>{brl(item['preco'])}</td>
            <td align='right'>{brl(item['qtd'] * item['preco'])}</td>
            <td>{item.get('data', '—')}</td>
        </tr>
        """
        for item in itens
    )

    return f"""
    <div style='font-family:monospace;max-width:520px;border:1px dashed #999;padding:12px'>
      <h3 style='margin:0'>🧾 RECIBO</h3>

      <table style='width:100%'>
        <tr>
          <th>Qtd.</th>
          <th>Evento</th>
          <th>Preço</th>
          <th>Total</th>
          <th>Data</th>
        </tr>

        {linhas}

      </table>

      <hr>

      <h3>TOTAL: {brl(total)}</h3>

      Pagamento: <b>{html.escape(pagamento)}</b><br>

      Observações: —
    </div>
    """


def estatisticas_html(tokens):
    contagem = Counter(t.type for t in tokens)
    barras = "".join(
        f"<div style='margin:2px 0'><code style='display:inline-block;width:120px'>{tipo}</code>"
        f"<span style='display:inline-block;background:{cor_do_token(tipo)};height:14px;width:{n*25}px'></span> {n}</div>"
        for tipo, n in contagem.most_common())
    return f"<h4>Frequência de cada categoria de token</h4>{barras}"

exemplos_a1 = {
    "Compra meias": '''# Compra #4821 — Sympla
INGRESSO 2x "Show Slayer" SETOR pista LOTE 2 MEIA NOME Lucas Carvalho R$ 200,00 EM 10/12/2026
INGRESSO 1x "Show Korn" SETOR banco LOTE 2 MEIA NOME Lucas Carvalho R$ 150,00 EM 10/12/2026
INGRESSO 3x "Show Slipknot" SETOR pista LOTE 3 MEIA NOME Lucas Carvalho R$ 1.500,00 EM 10/12/2026
PAGAMENTO pix''',
    "compra inteiras": '''# Festa da empresa — Pizzaria Bella
INGRESSO 2x "Show Slayer" SETOR pista LOTE 2 inteira NOME Lucas Carvalho R$ 200,00 EM 10/12/2026
INGRESSO 1x "Show Korn" SETOR banco LOTE 2 inteira NOME Lucas Carvalho R$ 150,00 EM 10/12/2026
INGRESSO 3x "Show Slipknot" SETOR pista LOTE 3 inteira NOME Lucas Carvalho R$ 1.500,00 EM 10/12/2026
pagamento cartão''',
    "❌ Erro: aspas não fechadas": '''INGRESSO 1x "Show Slayer R$ 19,90
PAGAMENTO cartão''',
    "❌ Erro: centavos com ponto": '''INGRESSO 2x "Show Slayer" 8.50''',
    "❌ Erro: tipo errado": '''INGRESSO 2x "Show Slayer" tipo meio''',
}

seletor_a2 = widgets.Dropdown(options=list(exemplos_a1), description="Exemplos:",
                              layout=widgets.Layout(width="60%"))
entrada_a2 = widgets.Textarea(value=exemplos_a1["Compra meias"],
                              layout=widgets.Layout(width="95%", height="190px"))
botao_a2 = widgets.Button(description="🔍 Analisar compra", button_style="success")
status_a2 = widgets.HTML()
abas_a2 = widgets.Tab(children=[widgets.Output() for _ in range(4)])
for i, nome in enumerate(["🎨 Colorido", "📋 Tokens", "🧾 Recibo", "📊 Estatística"]):
    abas_a2.set_title(i, nome)

def ao_escolher_a2(m):
    entrada_a2.value = exemplos_a1[m["new"]]
    ao_clicar_a2(None)

def ao_clicar_a2(_):
    texto = entrada_a2.value
    for aba in abas_a2.children:
        aba.clear_output()
    try:
        tokens = tokenizar_a1(texto)
    except UnexpectedCharacters as erro:
        status_a2.value = erro_lexico_html(texto, erro)
        return
    status_a2.value = f"<b style='color:#2e7d32'>✅ Análise léxica OK — {len(tokens)} tokens reconhecidos.</b>"
    conteudos = [texto_colorido_html(texto, tokens), tabela_tokens_html(tokens),
                 montar_recibo(tokens), estatisticas_html(tokens)]
    for aba, conteudo in zip(abas_a2.children, conteudos):
        with aba:
            display(HTML(conteudo))

seletor_a2.observe(ao_escolher_a2, names="value")
botao_a2.on_click(ao_clicar_a2)
display(widgets.HTML("<h3>🛵 Sistema de Pedidos — Analisador Léxico v2</h3>"),
        seletor_a2, entrada_a2, botao_a2, status_a2, abas_a2)
ao_clicar_a2(None)

HTML(value='<h3>🛵 Sistema de Pedidos — Analisador Léxico v2</h3>')

Dropdown(description='Exemplos:', layout=Layout(width='60%'), options=('Compra meias', 'compra inteiras', '❌ E…

Textarea(value='# Compra #4821 — Sympla\nINGRESSO 2x "Show Slayer" SETOR pista LOTE 2 MEIA NOME Lucas Carvalho…

Button(button_style='success', description='🔍 Analisar compra', style=ButtonStyle())

HTML(value='')